In [0]:
df = spark.sql("""
SELECT
  c.claim_id,
  c.claim_amount,
  c.LastUpdatedTimeStamp,
  c.policy_id,
  c.claim_status,
  TO_DATE(DATE_FORMAT(c.date_of_claim, 'MM-dd-yyyy'), 'MM-dd-yyyy') AS date_of_claim
FROM policyprojcatalog.policyprojdb.claim c
INNER JOIN policyprojcatalog.policyprojdb.policy p
  ON c.policy_id = p.policy_id
WHERE c.claim_amount IS NOT NULL
  AND c.claim_id IS NOT NULL
  AND c.policy_id IS NOT NULL
  AND c.claim_status IS NOT NULL
  AND c.LastUpdatedTimeStamp IS NOT NULL
  AND c.merge_flag = false
  AND c.claim_amount > 0
""")

display(df)

In [0]:
df.createOrReplaceTempView("clean_claim")

In [0]:
%sql
SELECT COUNT(*) FROM clean_claim;

In [0]:
%sql
MERGE INTO policyprojcatalog.silver.Claim AS T
USING clean_claim AS S
ON T.claim_id = S.claim_id

WHEN MATCHED THEN UPDATE SET
  T.claim_amount = S.claim_amount,
  T.LastUpdatedTimeStamp = S.LastUpdatedTimeStamp,
  T.policy_id = S.policy_id,
  T.claim_status = S.claim_status,
  T.date_of_claim = S.date_of_claim,
  T.merged_timestamp = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
  claim_id,
  claim_amount,
  LastUpdatedTimeStamp,
  policy_id,
  claim_status,
  date_of_claim,
  merged_timestamp
)
VALUES (
  S.claim_id,
  S.claim_amount,
  S.LastUpdatedTimeStamp,
  S.policy_id,
  S.claim_status,
  S.date_of_claim,
  current_timestamp()
);

In [0]:
spark.sql("""
UPDATE policyprojcatalog.policyprojdb.claim
SET merge_flag = true
WHERE merge_flag = false
""")

In [0]:
%sql
select * from policyprojcatalog.silver.claim

In [0]:
%sql
select * from policyprojcatalog.policyprojdb.claim

In [0]:
%sql
select * from policyprojcatalog.policyprojdb.policy

In [0]:
%sql
SELECT COUNT(*) FROM clean_claim;